# Cython

Мне влом писать про Cython, когда есть [документация](https://cython.readthedocs.io/en/latest/index.html) и [Wiki](https://github.com/cython/cython/wiki), так что здесь я оставлю разные примеры и необходимые пояснения.

Я заранее поставил в виртуальное окружение [cython](https://pypi.org/project/Cython/), поэтому мне не нужно морокаться с `%pip install cython`. Но установки Cython в окружение мало, нужно загрузить его как расширение:

In [1]:
%load_ext Cython

А теперь можно внести сам cython - как проверка наличия пакета это бессмысленно если расширение выше успешно загружено, но для возможного дальнейшего использования пусть будет.

In [2]:
import cython

Если выполнение успешно, то в ячейках тетрадки можно использовать `%%cython` - из-за `%%` команда `cython` применится ко всей ячейке, а не только лишь к строке как с `%`. Хотя есть простой пример из [документации](https://cython.readthedocs.io/en/latest/src/quickstart/build.html), но лучше покрутить на уже знакомом и начнё с факториала:

In [3]:
%%cython -a
# -a/--annotate -> gen ad-hoc HTML with underlying transformations into C

cimport cython

# def - accessible in Python-space -> "Pythonic" function anyway
# `int n` is Cython, in Python would be `n: int`
#@cython.overflowcheck(True)
def factorial_cy1(int n):  # no need to use an annotation for a return value
    if n < 0:
        msg = f"{n} < 0"
        raise ValueError(msg) from None
    if n < 2:
        return 1
    cdef int i  # declare an integer `i` -> `int i`
    cdef long long f = 1  # initialise like `long long int s = 1;` 
    for i in range(2, n + 1):  # n is not expected to be overflown
        # No need for an `overflowcheck` decorator,
        # because I expect the overflow only for `f` here
        with cython.overflowcheck(True):
            f *= i
    return f

try:
    factorial_cy1(-1)
except ValueError as e:
    print(e)

for n in range(0, 21):
    print(f"{n:>2}! -> {factorial_cy1(n)}")

try:
    factorial_cy1(21)
except OverflowError as e:
    print(f"21 -> {e}")

-1 < 0
 0! -> 1
 1! -> 1
 2! -> 2
 3! -> 6
 4! -> 24
 5! -> 120
 6! -> 720
 7! -> 5040
 8! -> 40320
 9! -> 362880
10! -> 3628800
11! -> 39916800
12! -> 479001600
13! -> 6227020800
14! -> 87178291200
15! -> 1307674368000
16! -> 20922789888000
17! -> 355687428096000
18! -> 6402373705728000
19! -> 121645100408832000
20! -> 2432902008176640000


На моей машине (Ubuntu 24 LTS) переполнение наступает уже для `n = 21`: таковы ограничения типов и "long long" просто не вместил значение. Я задумал функцию именно такой чтобы показать случай переполнения типа. Поскольку на разных платформах размеры типы отличаются, то не следует привязываться к числу 21 и делать "костыли" исходя из этой границы (если только именно в этом нет насущной потребности).

И всё же это ограничение так себе, а хочется позволить работать с большими числами. Оказывается, достаточно обойтись малыми поменами:

In [4]:
%%cython --annotate

cimport cython

@cython.overflowcheck(True)
def factorial_cy2(int n):
    if n < 0:
        msg = f"{n} < 0"
        raise ValueError(msg) from None
    if n < 2:
        return 1
    cdef int i  # declare an integer `i` -> `int i`
    # f = 1  # Python object if Cython infer_types is None
    cdef object f = 1  # treat f as Python object explicitly and give no damn about infer_types 
    for i in range(2, n + 1):
        f *= i
    return f


for n in [20, 21, 22]:
    print(f"{n:>2}! -> {factorial_cy2(n)}")

20! -> 2432902008176640000
21! -> 51090942171709440000
22! -> 1124000727777607680000


Достаточно убрать ограничение `cdef long long f = 1` и рассматривать f как Python объект: чуть менее производительно, зато можно считать большие числа и уже это может быть оправданно выгодно. И потом, некоторые части остались, тот же цикл идёт в Си код и не все улучшения потеряны (пока нет нужды углублять в это, можно "просто так чувствовать").